<a href="https://colab.research.google.com/github/EmanueleDeCandia/Logistica-Vending-Machine/blob/main/Monte_Carlo_OCS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monte Carlo OCS: Analisi di Rischio e Redditività

## Utilizzo
È possibile aggiungere tutti gli addetti necessari. La regione viene calcolata sommando, in ogni iterazione Monte Carlo, i risultati dei singoli portafogli.

Questo permette di individuare:
* Addetti con portafogli strutturalmente meno redditizi
* Differenze di densità logistica
* Rischio di sovraccarico
* Costo per cliente
* Valore e margine prodotto da ciascun addetto
* Probabilità di perdita del singolo portafoglio
* Rischio complessivo regionale

La scelta delle variabili riflette componenti rilevanti nell’OCS: densità dei percorsi, affidabilità delle macchine, inflazione e livelli di servizio.

---

## Variabili Stocastiche Inserite

### 1. Domanda e Profittabilità
* Variazione del numero di clienti attivi
* Variazione del valore annuo medio del cliente
* Ordini in meno, sconti, insoluti e rettifiche
* Perdita di ricavo durante il fermo macchina

### 2. Logistica
* Visite extra (ordini incompleti o urgenti)
* Clienti last-minute e sforamento chilometri
* Durata del servizio e inflazione costi chilometrici

### 3. Personale
* Giorni di assenza e costi di sostituzione
* Straordinari e saturazione oraria

### 4. Macchine OCS e Prodotti
* Giorni di fermo, guasti e costi di intervento
* Incidenza costo prodotti e relativa inflazione

---

## Controllo dell’Incertezza

Ogni variabile è configurata con una distribuzione gaussiana troncata:

```json
"visite_extra_ordini_incompleti_pct": {
    "mean": 0.055,
    "sd": 0.030,
    "min": 0.00,
    "max": 0.20
}
```

È possibile scalare l'incertezza globale tramite il parametro `UNCERTAINTY_MULTIPLIER` (es. `0.50` per dimezzarla, `2.00` per raddoppiarla).

### Variabili Opzionali
Alcune variabili (come il *churn* o i tempi di parcheggio) sono predisposte nel codice ma disattivate con `#`. Per attivarle, è necessario rimuovere il commento sia nella configurazione `SHOCKS` che nella funzione `simula_addetto()`.

---

## Risultati della Prova Tecnica (Esempio)

Con 10.000 simulazioni e parametri dimostrativi:

| Indicatore | Risultato |
| :--- | :--- |
| **Margine medio regionale** | € 13.572 |
| **Margine P5 (Rischio)** | -€ 28.927 |
| **Margine mediano** | € 13.310 |
| **Margine P95 (Opportunità)** | € 56.821 |
| **Probabilità di perdita** | 29,98% |
| **ROI medio** | 2,47% |
| **Probabilità di sovraccarico** | 85,29% |

> **Nota:** Questi risultati sono puramente esemplificativi e dipendono dai parametri di input inseriti.

L'output Excel generato contiene i fogli di dettaglio per input, sintesi area/addetti, analisi di sensibilità e simulazioni complete.

### Analisi Tecnica del Modello di Simulazione

Il codice implementa un motore di simulazione **Monte Carlo** personalizzato per il settore OCS. Ecco i pilastri logici del primo blocco:

#### 1. Configurazione e Portafogli
Il modello definisce parametri operativi standard (velocità media, ore lavorative, giorni annui) e accetta una lista di `PORTAFOGLI_ADDETTI`. Ogni portafoglio rappresenta un'entità economica separata con i propri clienti base, chilometri percorsi e costi fissi.

#### 2. Motore degli Shock (Incertezza)
Il cuore della simulazione risiede nel dizionario `SHOCKS`. Ogni variabile (es. inflazione, guasti, variazioni di volume) è definita come una distribuzione **Gaussiana Troncata**:
* **Media (`mean`)**: Il valore atteso.
* **Deviazione Standard (`sd`)**: L'ampiezza dell'incertezza.
* **Limiti (`min`/`max`)**: Impediscono valori irrealistici (es. costi negativi o inflazione infinita).
* **`UNCERTAINTY_MULTIPLIER`**: Un parametro globale per testare scenari di stress (es. raddoppiando l'incertezza).

#### 3. Logica di Simulazione (`simula_portafoglio_addetto`)
Per ogni iterazione (default 10.000), il codice:
1. **Estrae valori casuali** per tutti i driver di rischio.
2. **Calcola l'impatto a cascata**: Ad esempio, un aumento di clienti impatta i chilometri, che impattano il tempo di guida, che può generare straordinari (costo del personale).
3. **Determina i Margini**: Sottrae dai ricavi netti (corretti per fermi macchina e insoluti) i costi operativi (logistica, prodotto, manutenzione, personale).

#### 4. Aggregazione Regionale
La funzione `simula_area_regionale` coordina l'esecuzione. Gli shock "macro" (come l'inflazione dei prodotti) sono mantenuti coerenti per tutti gli addetti nella stessa iterazione, simulando un mercato reale dove certi rischi colpiscono l'intera regione simultaneamente.

In [4]:
import os
# Crea la cartella mancante richiesta dal tuo codice
os.makedirs('/mnt/data', exist_ok=True)
print("Cartella /mnt/data creata o già esistente.")

Cartella /mnt/data creata o già esistente.


In [5]:
from pathlib import Path

code = r'''
"""
Monte Carlo OCS: rischio logistico e redditività
================================================
Analizza:
- una regione, aggregando i portafogli reali degli addetti;
- ogni singolo portafoglio/addetto.

Usa gaussiane troncate. Modificare media, sd, min e max in SHOCKS.
UNCERTAINTY_MULTIPLIER scala contemporaneamente tutte le sd.

Dipendenze:
    pip install numpy pandas matplotlib openpyxl
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
N_SIMULATIONS = 10_000
UNCERTAINTY_MULTIPLIER = 1.0

GIORNI_LAVORATIVI_MESE = 22
MESI_ANNO = 12
GIORNI_LAVORATIVI_ANNO = GIORNI_LAVORATIVI_MESE * MESI_ANNO
ORE_LAVORATIVE_GIORNO = 8.0
VELOCITA_MEDIA_KMH = 42.0

OUTPUT_DIR = Path("output_monte_carlo_ocs")

# Ogni record è il gruppo reale di clienti coperto da un addetto.
PORTAFOGLI_ADDETTI = [
    {
        "area": "Marche",
        "addetto": "Addetto_01",
        "clienti_base": 170,
        "km_annui_base": 50_000,
        "valore_annuo_cliente_base": 1_250,
        "frequenza_mensile_base": 1.0,
        "costo_lavoro_annuo_base": 36_000,
        "costo_km_base": 0.85,
        "costo_manutenzione_macchina_base": 85,
        "macchine_per_cliente": 1.0,
    },
    {
        "area": "Marche",
        "addetto": "Addetto_02",
        "clienti_base": 165,
        "km_annui_base": 48_000,
        "valore_annuo_cliente_base": 1_180,
        "frequenza_mensile_base": 1.0,
        "costo_lavoro_annuo_base": 36_000,
        "costo_km_base": 0.85,
        "costo_manutenzione_macchina_base": 85,
        "macchine_per_cliente": 1.0,
    },
    {
        "area": "Marche",
        "addetto": "Addetto_03",
        "clienti_base": 165,
        "km_annui_base": 52_000,
        "valore_annuo_cliente_base": 1_220,
        "frequenza_mensile_base": 1.0,
        "costo_lavoro_annuo_base": 36_000,
        "costo_km_base": 0.85,
        "costo_manutenzione_macchina_base": 85,
        "macchine_per_cliente": 1.0,
    },
]

# Percentuali in forma decimale: 0.05 = 5%.
SHOCKS = {
    # DOMANDA E CLIENTI
    "variazione_clienti_pct": dict(mean=0.00, sd=0.035, min=-0.12, max=0.12),
    "variazione_valore_cliente_pct": dict(mean=0.00, sd=0.08, min=-0.25, max=0.30),
    "ordini_in_meno_pct": dict(mean=0.025, sd=0.018, min=0.00, max=0.12),
    "rettifiche_ricavi_pct": dict(mean=0.015, sd=0.010, min=0.00, max=0.07),

    # VISITE E PERCORSI
    "visite_extra_ordini_incompleti_pct": dict(mean=0.055, sd=0.030, min=0.00, max=0.20),
    "visite_urgenti_pct": dict(mean=0.025, sd=0.018, min=0.00, max=0.12),
    "sforamento_km_percorso_pct": dict(mean=0.045, sd=0.035, min=-0.04, max=0.22),
    "km_extra_per_visita": dict(mean=18.0, sd=6.0, min=3.0, max=45.0),
    "minuti_servizio_per_visita": dict(mean=24.0, sd=5.0, min=10.0, max=50.0),

    # PERSONALE
    "giorni_assenza_addetto": dict(mean=8.0, sd=4.0, min=0.0, max=30.0),
    "costo_sostituzione_giorno": dict(mean=185.0, sd=30.0, min=100.0, max=300.0),
    "costo_straordinario_ora": dict(mean=28.0, sd=4.0, min=18.0, max=45.0),

    # MACCHINE OCS
    "giorni_fermo_macchina_per_cliente": dict(mean=2.0, sd=1.2, min=0.0, max=8.0),
    "quota_ricavo_persa_durante_fermo": dict(mean=0.65, sd=0.12, min=0.20, max=1.00),
    "guasti_straordinari_per_100_macchine": dict(mean=9.0, sd=3.0, min=1.0, max=25.0),
    "costo_medio_intervento_guasto": dict(mean=145.0, sd=40.0, min=50.0, max=350.0),

    # PRODOTTI E LOGISTICA
    "costo_prodotti_pct_ricavi": dict(mean=0.41, sd=0.045, min=0.28, max=0.58),
    "inflazione_prodotti_pct": dict(mean=0.035, sd=0.025, min=-0.03, max=0.15),
    "inflazione_costo_km_pct": dict(mean=0.035, sd=0.030, min=-0.06, max=0.18),
    "variazione_manutenzione_pct": dict(mean=0.00, sd=0.12, min=-0.20, max=0.40),

    # VARIABILI OPZIONALI
    # Togliere # qui e nel blocco omonimo dentro simula_addetto().
    #
    # "churn_servizio_pct": dict(mean=0.012, sd=0.010, min=0.00, max=0.06),
    # "ritardo_accesso_minuti": dict(mean=4.0, sd=3.0, min=0.0, max=18.0),
    # "sprechi_prodotti_pct": dict(mean=0.010, sd=0.008, min=0.00, max=0.05),
    # "costo_extra_igiene_per_macchina": dict(mean=22.0, sd=8.0, min=5.0, max=55.0),
}


def gauss(rng, cfg, n):
    values = rng.normal(cfg["mean"], cfg["sd"] * UNCERTAINTY_MULTIPLIER, n)
    return np.clip(values, cfg["min"], cfg["max"])


def draw_all(rng, n):
    return {name: gauss(rng, cfg, n) for name, cfg in SHOCKS.items()}


def div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.divide(a, b, out=np.zeros_like(a), where=b != 0)


def simula_addetto(cfg, macro, rng, n):
    s = draw_all(rng, n)
    s["inflazione_prodotti_pct"] = macro["inflazione_prodotti_pct"]
    s["inflazione_costo_km_pct"] = macro["inflazione_costo_km_pct"]

    clienti = np.maximum(
        np.rint(cfg["clienti_base"] * (1 + s["variazione_clienti_pct"])),
        1,
    )

    # OPZIONALE: churn legato al servizio.
    # clienti -= np.rint(clienti * s["churn_servizio_pct"])
    # clienti = np.maximum(clienti, 1)

    macchine = clienti * cfg["macchine_per_cliente"]
    valore_cliente = cfg["valore_annuo_cliente_base"] * (
        1 + s["variazione_valore_cliente_pct"]
    )
    ricavi_lordi = clienti * valore_cliente
    perdita_ordini = ricavi_lordi * s["ordini_in_meno_pct"]
    perdita_fermo = (
        ricavi_lordi
        * s["giorni_fermo_macchina_per_cliente"]
        / 365
        * s["quota_ricavo_persa_durante_fermo"]
    )
    rettifiche = ricavi_lordi * s["rettifiche_ricavi_pct"]
    ricavi_netti = np.maximum(
        ricavi_lordi - perdita_ordini - perdita_fermo - rettifiche,
        0,
    )

    visite_programmate = (
        clienti * cfg["frequenza_mensile_base"] * MESI_ANNO
    )
    visite_extra = visite_programmate * s["visite_extra_ordini_incompleti_pct"]
    visite_urgenti = visite_programmate * s["visite_urgenti_pct"]
    visite_totali = visite_programmate + visite_extra + visite_urgenti

    km_ordinari = (
        cfg["km_annui_base"]
        * (clienti / cfg["clienti_base"])
        * (1 + s["sforamento_km_percorso_pct"])
    )
    km_extra = (visite_extra + visite_urgenti) * s["km_extra_per_visita"]
    km_totali = np.maximum(km_ordinari + km_extra, 0)

    minuti_visita = s["minuti_servizio_per_visita"]

    # OPZIONALE: difficoltà di accesso, parcheggio e referente assente.
    # minuti_visita += s["ritardo_accesso_minuti"]

    ore_servizio = visite_totali * minuti_visita / 60
    ore_guida = km_totali / VELOCITA_MEDIA_KMH
    ore_richieste = ore_servizio + ore_guida

    giorni_disponibili = np.maximum(
        GIORNI_LAVORATIVI_ANNO - s["giorni_assenza_addetto"],
        1,
    )
    ore_disponibili = giorni_disponibili * ORE_LAVORATIVE_GIORNO
    straordinario = np.maximum(ore_richieste - ore_disponibili, 0)
    saturazione = div(ore_richieste, ore_disponibili)

    costo_km = cfg["costo_km_base"] * (1 + s["inflazione_costo_km_pct"])
    costo_logistica = km_totali * costo_km

    costo_prodotti_ratio = np.clip(
        s["costo_prodotti_pct_ricavi"] * (1 + s["inflazione_prodotti_pct"]),
        0,
        0.90,
    )
    costo_prodotti = ricavi_netti * costo_prodotti_ratio

    costo_personale = (
        cfg["costo_lavoro_annuo_base"]
        + s["giorni_assenza_addetto"] * s["costo_sostituzione_giorno"]
        + straordinario * s["costo_straordinario_ora"]
    )

    manutenzione = (
        macchine
        * cfg["costo_manutenzione_macchina_base"]
        * (1 + s["variazione_manutenzione_pct"])
    )
    numero_guasti = (
        macchine * s["guasti_straordinari_per_100_macchine"] / 100
    )
    costo_guasti = numero_guasti * s["costo_medio_intervento_guasto"]

    # OPZIONALE: sprechi e scadenze.
    # costo_sprechi = costo_prodotti * s["sprechi_prodotti_pct"]
    costo_sprechi = np.zeros(n)

    # OPZIONALE: igiene, filtri acqua e sanificazioni aggiuntive.
    # costo_extra_igiene = macchine * s["costo_extra_igiene_per_macchina"]
    costo_extra_igiene = np.zeros(n)

    costi_totali = (
        costo_prodotti
        + costo_logistica
        + costo_personale
        + manutenzione
        + costo_guasti
        + costo_sprechi
        + costo_extra_igiene
    )
    margine = ricavi_netti - costi_totali
    roi = div(margine, costi_totali) * 100

    return pd.DataFrame({
        "simulazione": np.arange(1, n + 1),
        "area": cfg["area"],
        "addetto": cfg["addetto"],

        "shock_variazione_clienti_pct": s["variazione_clienti_pct"],
        "shock_variazione_valore_cliente_pct": s["variazione_valore_cliente_pct"],
        "shock_ordini_in_meno_pct": s["ordini_in_meno_pct"],
        "shock_visite_extra_pct": s["visite_extra_ordini_incompleti_pct"],
        "shock_visite_urgenti_pct": s["visite_urgenti_pct"],
        "shock_sforamento_km_pct": s["sforamento_km_percorso_pct"],
        "shock_giorni_assenza": s["giorni_assenza_addetto"],
        "shock_giorni_fermo_macchina": s["giorni_fermo_macchina_per_cliente"],
        "shock_inflazione_prodotti_pct": s["inflazione_prodotti_pct"],
        "shock_inflazione_costo_km_pct": s["inflazione_costo_km_pct"],
        "shock_costo_prodotti_pct": s["costo_prodotti_pct_ricavi"],

        "clienti_attivi": clienti,
        "macchine_attive": macchine,
        "visite_programmate_anno": visite_programmate,
        "visite_extra_anno": visite_extra,
        "visite_urgenti_anno": visite_urgenti,
        "visite_totali_anno": visite_totali,
        "visite_mese": visite_totali / MESI_ANNO,
        "visite_giorno": visite_totali / GIORNI_LAVORATIVI_ANNO,
        "km_totali": km_totali,
        "km_per_visita": div(km_totali, visite_totali),
        "ore_richieste": ore_richieste,
        "ore_disponibili": ore_disponibili,
        "ore_straordinario": straordinario,
        "saturazione_operativa": saturazione,
        "sovraccarico_operativo": saturazione > 1,

        "ricavi_lordi": ricavi_lordi,
        "perdita_ordini": perdita_ordini,
        "perdita_ricavi_fermo": perdita_fermo,
        "rettifiche_ricavi": rettifiche,
        "ricavi_netti": ricavi_netti,
        "costo_prodotti": costo_prodotti,
        "costo_logistica": costo_logistica,
        "costo_personale": costo_personale,
        "costo_manutenzione_preventiva": manutenzione,
        "costo_guasti": costo_guasti,
        "costi_operativi_totali": costi_totali,
        "margine_netto_operativo": margine,
        "roi_operativo_pct": roi,
        "valore_per_addetto": ricavi_netti,
        "ricavo_per_cliente": div(ricavi_netti, clienti),
        "costo_per_cliente": div(costi_totali, clienti),
        "margine_per_cliente": div(margine, clienti),
        "costo_per_visita": div(costi_totali, visite_totali),
        "margine_per_visita": div(margine, visite_totali),
    })


def simula_regione(portafogli, n=N_SIMULATIONS, seed=SEED):
    rng = np.random.default_rng(seed)
    macro = {
        "inflazione_prodotti_pct": gauss(
            rng, SHOCKS["inflazione_prodotti_pct"], n
        ),
        "inflazione_costo_km_pct": gauss(
            rng, SHOCKS["inflazione_costo_km_pct"], n
        ),
    }

    dettaglio = pd.concat(
        [simula_addetto(p, macro, rng, n) for p in portafogli],
        ignore_index=True,
    )

    sum_cols = [
        "clienti_attivi", "macchine_attive", "visite_programmate_anno",
        "visite_extra_anno", "visite_urgenti_anno", "visite_totali_anno",
        "km_totali", "ore_richieste", "ore_disponibili",
        "ore_straordinario", "ricavi_lordi", "perdita_ordini",
        "perdita_ricavi_fermo", "rettifiche_ricavi", "ricavi_netti",
        "costo_prodotti", "costo_logistica", "costo_personale",
        "costo_manutenzione_preventiva", "costo_guasti",
        "costi_operativi_totali", "margine_netto_operativo",
    ]
    area = (
        dettaglio.groupby(["area", "simulazione"], as_index=False)[sum_cols]
        .sum()
    )
    n_addetti = len(portafogli)
    area["numero_addetti"] = n_addetti
    area["clienti_per_addetto"] = area["clienti_attivi"] / n_addetti
    area["valore_per_addetto"] = area["ricavi_netti"] / n_addetti
    area["visite_giorno_per_addetto"] = (
        area["visite_totali_anno"] / GIORNI_LAVORATIVI_ANNO / n_addetti
    )
    area["km_per_addetto"] = area["km_totali"] / n_addetti
    area["km_per_cliente"] = div(area["km_totali"], area["clienti_attivi"])
    area["km_per_visita"] = div(area["km_totali"], area["visite_totali_anno"])
    area["ricavo_per_cliente"] = div(area["ricavi_netti"], area["clienti_attivi"])
    area["costo_per_cliente"] = div(
        area["costi_operativi_totali"], area["clienti_attivi"]
    )
    area["margine_per_cliente"] = div(
        area["margine_netto_operativo"], area["clienti_attivi"]
    )
    area["roi_operativo_pct"] = div(
        area["margine_netto_operativo"], area["costi_operativi_totali"]
    ) * 100
    area["saturazione_operativa"] = div(
        area["ore_richieste"], area["ore_disponibili"]
    )
    area["sovraccarico_operativo"] = area["saturazione_operativa"] > 1

    drivers = [
        "shock_variazione_clienti_pct",
        "shock_variazione_valore_cliente_pct",
        "shock_ordini_in_meno_pct",
        "shock_visite_extra_pct",
        "shock_visite_urgenti_pct",
        "shock_sforamento_km_pct",
        "shock_giorni_assenza",
        "shock_giorni_fermo_macchina",
        "shock_inflazione_prodotti_pct",
        "shock_inflazione_costo_km_pct",
        "shock_costo_prodotti_pct",
    ]
    avg_drivers = (
        dettaglio.groupby(["area", "simulazione"], as_index=False)[drivers]
        .mean()
    )
    area = area.merge(avg_drivers, on=["area", "simulazione"])
    return dettaglio, area


def summary_table(df, group, indicators):
    rows = []
    for name, data in df.groupby(group):
        for col in indicators:
            x = data[col].dropna()
            rows.append({
                group: name,
                "indicatore": col,
                "P5": x.quantile(0.05),
                "media": x.mean(),
                "mediana_P50": x.median(),
                "P95": x.quantile(0.95),
                "deviazione_standard": x.std(),
                "probabilita_negativo": (x < 0).mean(),
            })
    return pd.DataFrame(rows)


def addetti_summary(df):
    rows = []
    for name, x in df.groupby("addetto"):
        rows.append({
            "addetto": name,
            "clienti_media": x["clienti_attivi"].mean(),
            "valore_per_addetto_media": x["ricavi_netti"].mean(),
            "margine_medio": x["margine_netto_operativo"].mean(),
            "margine_P5": x["margine_netto_operativo"].quantile(0.05),
            "margine_P50": x["margine_netto_operativo"].median(),
            "margine_P95": x["margine_netto_operativo"].quantile(0.95),
            "probabilita_perdita": (x["margine_netto_operativo"] < 0).mean(),
            "roi_medio_pct": x["roi_operativo_pct"].mean(),
            "km_medi": x["km_totali"].mean(),
            "visite_giorno_medie": x["visite_giorno"].mean(),
            "saturazione_media": x["saturazione_operativa"].mean(),
            "probabilita_sovraccarico": x["sovraccarico_operativo"].mean(),
            "costo_cliente_medio": x["costo_per_cliente"].mean(),
            "margine_cliente_medio": x["margine_per_cliente"].mean(),
        })
    return pd.DataFrame(rows)


def sensitivity(area, target="margine_netto_operativo"):
    drivers = [
        "shock_variazione_clienti_pct",
        "shock_variazione_valore_cliente_pct",
        "shock_ordini_in_meno_pct",
        "shock_visite_extra_pct",
        "shock_visite_urgenti_pct",
        "shock_sforamento_km_pct",
        "shock_giorni_assenza",
        "shock_giorni_fermo_macchina",
        "shock_inflazione_prodotti_pct",
        "shock_inflazione_costo_km_pct",
        "shock_costo_prodotti_pct",
    ]
    s = (
        area[drivers + [target]].corr(method="spearman")[target]
        .drop(target)
        .sort_values(key=np.abs, ascending=False)
    )
    return s.rename("correlazione_spearman").reset_index(
        names="variabile"
    )


def save_charts(area, detail, sens, out):
    paths = []

    x = area["margine_netto_operativo"]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(x, bins=35, alpha=0.75, edgecolor="black")
    ax.axvline(x.mean(), linestyle="--", label=f"Media: €{x.mean():,.0f}")
    ax.axvline(x.quantile(.05), linestyle=":", label=f"P5: €{x.quantile(.05):,.0f}")
    ax.set(title="Distribuzione Monte Carlo del Margine Netto OCS",
           xlabel="Margine netto operativo annuo (€)",
           ylabel="Numero di simulazioni")
    ax.legend()
    fig.tight_layout()
    p = out / "01_distribuzione_margine.png"
    fig.savefig(p, dpi=180, bbox_inches="tight")
    plt.close(fig); paths.append(p)

    x = area["roi_operativo_pct"]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(x, bins=35, alpha=0.75, edgecolor="black")
    ax.axvline(x.mean(), linestyle="--", label=f"Media: {x.mean():.1f}%")
    ax.axvline(x.quantile(.05), linestyle=":", label=f"P5: {x.quantile(.05):.1f}%")
    ax.set(title="Distribuzione Monte Carlo del ROI Operativo OCS",
           xlabel="ROI operativo (%)",
           ylabel="Numero di simulazioni")
    ax.legend()
    fig.tight_layout()
    p = out / "02_distribuzione_roi.png"
    fig.savefig(p, dpi=180, bbox_inches="tight")
    plt.close(fig); paths.append(p)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(area["km_totali"], area["margine_netto_operativo"],
               alpha=.25, s=16)
    ax.set(title="Impatto dei Km Totali sul Margine Netto",
           xlabel="Km annui della flotta",
           ylabel="Margine netto operativo (€)")
    fig.tight_layout()
    p = out / "03_impatto_km_margine.png"
    fig.savefig(p, dpi=180, bbox_inches="tight")
    plt.close(fig); paths.append(p)

    d = sens.sort_values("correlazione_spearman")
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(d["variabile"], d["correlazione_spearman"])
    ax.axvline(0, linewidth=1)
    ax.set(title="Sensibilità del Margine Netto ai Driver Stocastici",
           xlabel="Correlazione di Spearman con il margine",
           ylabel="Variabile")
    fig.tight_layout()
    p = out / "04_sensibilita_margine.png"
    fig.savefig(p, dpi=180, bbox_inches="tight")
    plt.close(fig); paths.append(p)

    groups = [
        g["margine_netto_operativo"].to_numpy()
        for _, g in detail.groupby("addetto")
    ]
    labels = [name for name, _ in detail.groupby("addetto")]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.boxplot(groups, tick_labels=labels, showfliers=False)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set(title="Confronto della Distribuzione del Margine per Addetto",
           xlabel="Portafoglio/addetto",
           ylabel="Margine netto operativo (€)")
    fig.tight_layout()
    p = out / "05_confronto_addetti.png"
    fig.savefig(p, dpi=180, bbox_inches="tight")
    plt.close(fig); paths.append(p)

    return paths


def export_excel(detail, area, area_summary, operator_summary, sens, out):
    path = out / "risultati_monte_carlo_ocs.xlsx"
    shock_cfg = pd.DataFrame([
        {
            "variabile": name,
            "media": cfg["mean"],
            "deviazione_standard": cfg["sd"],
            "minimo": cfg["min"],
            "massimo": cfg["max"],
            "moltiplicatore_incertezza": UNCERTAINTY_MULTIPLIER,
        }
        for name, cfg in SHOCKS.items()
    ])
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        pd.DataFrame(PORTAFOGLI_ADDETTI).to_excel(
            writer, sheet_name="Input_addetti", index=False
        )
        shock_cfg.to_excel(
            writer, sheet_name="Input_incertezza", index=False
        )
        area_summary.to_excel(
            writer, sheet_name="Sintesi_area", index=False
        )
        operator_summary.to_excel(
            writer, sheet_name="Sintesi_addetti", index=False
        )
        sens.to_excel(writer, sheet_name="Sensibilita", index=False)
        area.to_excel(writer, sheet_name="Simulazioni_area", index=False)
        detail.to_excel(
            writer, sheet_name="Simulazioni_addetti", index=False
        )
    return path


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    detail, area = simula_regione(PORTAFOGLI_ADDETTI)

    indicators = [
        "clienti_attivi", "ricavi_netti", "costi_operativi_totali",
        "margine_netto_operativo", "roi_operativo_pct", "km_totali",
        "visite_giorno_per_addetto", "costo_per_cliente",
        "margine_per_cliente", "saturazione_operativa",
    ]
    area_summary = summary_table(area, "area", indicators)
    operator_summary = addetti_summary(detail)
    sens = sensitivity(area)
    charts = save_charts(area, detail, sens, OUTPUT_DIR)
    excel = export_excel(
        detail, area, area_summary, operator_summary, sens, OUTPUT_DIR
    )

    margin = area["margine_netto_operativo"]
    roi = area["roi_operativo_pct"]

    print("=" * 72)
    print("MONTE CARLO OCS — RISULTATI AREA REGIONALE")
    print("=" * 72)
    print(f"Simulazioni: {N_SIMULATIONS:,}")
    print(f"Area: {area['area'].iloc[0]}")
    print(f"Addetti: {len(PORTAFOGLI_ADDETTI)}")
    print(f"Clienti base: {sum(p['clienti_base'] for p in PORTAFOGLI_ADDETTI):,.0f}")
    print()
    print("MARGINE NETTO")
    print(f"Media:   € {margin.mean():,.2f}")
    print(f"P5:      € {margin.quantile(.05):,.2f}")
    print(f"Mediana: € {margin.median():,.2f}")
    print(f"P95:     € {margin.quantile(.95):,.2f}")
    print(f"Probabilità di perdita: {(margin < 0).mean():.2%}")
    print()
    print("ROI")
    print(f"Media: {roi.mean():.2f}%")
    print(f"P5: {roi.quantile(.05):.2f}%")
    print(f"P95: {roi.quantile(.95):.2f}%")
    print(
        "Probabilità di sovraccarico: "
        f"{area['sovraccarico_operativo'].mean():.2%}"
    )
    print()
    print("PRINCIPALI DRIVER")
    print(sens.head(8).to_string(index=False))
    print()
    print(f"Excel: {excel.resolve()}")
    for p in charts:
        print(f"Grafico: {p.resolve()}")


if __name__ == "__main__":
    main()
'''

from pathlib import Path
path = Path("/mnt/data/monte_carlo_ocs.py")
path.write_text(code, encoding="utf-8")
print(path)
print(len(code.splitlines()))


/mnt/data/monte_carlo_ocs.py
597


In [6]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [7]:
from pathlib import Path

code = r'''"""
MONTE CARLO OCS — RISCHIO LOGISTICO E REDDITIVITÀ
==================================================

Ambito:
    Office Coffee Service (OCS) per caffè in ufficio.
    Non è un modello per distributori automatici tradizionali.

Obiettivo:
    Valutare l'incertezza economica e operativa:
    - dell'intera area regionale;
    - del portafoglio clienti assegnato a ciascun addetto.

Il modello non usa griglie di scenari fissi. Ogni simulazione estrae valori
casuali da distribuzioni gaussiane troncate, definite tramite:
    - media;
    - deviazione standard;
    - limite minimo;
    - limite superiore.

La deviazione standard di ogni variabile è modificabile. Inoltre:
    UNCERTAINTY_MULTIPLIER
permette di aumentare o ridurre simultaneamente tutte le deviazioni standard.

Output:
    - dettaglio di ogni simulazione per area;
    - dettaglio di ogni simulazione per addetto;
    - sintesi P5, media, mediana e P95;
    - probabilità di perdita;
    - probabilità di sovraccarico operativo;
    - analisi di sensibilità;
    - grafici separati;
    - file Excel.

Dipendenze:
    pip install numpy pandas matplotlib openpyxl
"""

from __future__ import annotations
from IPython.display import display
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =====================================================================
# 1. CONFIGURAZIONE GENERALE
# =====================================================================

SEED = 42
N_SIMULATIONS = 10_000

GIORNI_LAVORATIVI_MESE = 22
MESI_ANNO = 12
GIORNI_LAVORATIVI_ANNO = GIORNI_LAVORATIVI_MESE * MESI_ANNO

ORE_LAVORATIVE_GIORNO = 8.0
VELOCITA_MEDIA_KMH = 42.0

# Moltiplica tutte le deviazioni standard.
# 0.50 = incertezza dimezzata
# 1.00 = incertezza impostata nelle singole variabili
# 1.50 = incertezza aumentata del 50%
# 2.00 = deviazioni standard raddoppiate
UNCERTAINTY_MULTIPLIER = 1.00

OUTPUT_DIR = Path("output_monte_carlo_ocs")


# =====================================================================
# 2. PORTAFOGLI REALI DEGLI ADDETTI
# =====================================================================
#
# Ogni elemento rappresenta il gruppo reale di clienti assegnato a un addetto.
# La regione è ottenuta sommando i risultati di tutti gli addetti.
#
# È possibile:
# - aggiungere altri addetti;
# - eliminare righe;
# - assegnare parametri diversi a ogni portafoglio;
# - usare un solo elemento per analizzare un singolo addetto.

PORTAFOGLI_ADDETTI = [
    {
        "area": "Marche",
        "addetto": "Addetto_01",
        "clienti_base": 170,
        "km_annui_base": 50_000,
        "valore_annuo_cliente_base": 1_250,
        "frequenza_mensile_base": 1.00,
        "costo_lavoro_annuo_base": 36_000,
        "costo_km_base": 0.85,
        "costo_manutenzione_macchina_base": 85,
        "macchine_per_cliente": 1.00,
    },
    {
        "area": "Marche",
        "addetto": "Addetto_02",
        "clienti_base": 165,
        "km_annui_base": 48_000,
        "valore_annuo_cliente_base": 1_180,
        "frequenza_mensile_base": 1.00,
        "costo_lavoro_annuo_base": 36_000,
        "costo_km_base": 0.85,
        "costo_manutenzione_macchina_base": 85,
        "macchine_per_cliente": 1.00,
    },
    {
        "area": "Marche",
        "addetto": "Addetto_03",
        "clienti_base": 165,
        "km_annui_base": 52_000,
        "valore_annuo_cliente_base": 1_220,
        "frequenza_mensile_base": 1.00,
        "costo_lavoro_annuo_base": 36_000,
        "costo_km_base": 0.85,
        "costo_manutenzione_macchina_base": 85,
        "macchine_per_cliente": 1.00,
    },
]


# =====================================================================
# 3. DISTRIBUZIONI GAUSSIANE DEGLI SHOCK
# =====================================================================
#
# Formato di ogni variabile:
# {
#     "mean": media,
#     "sd": deviazione standard,
#     "min": limite inferiore,
#     "max": limite superiore,
# }
#
# Le percentuali sono espresse in forma decimale:
#     0.05 = +5%
#    -0.05 = -5%
#
# Alcune variabili opzionali sono commentate con #.
# Per attivarle:
# 1. rimuovere il simbolo # dalla configurazione;
# 2. rimuovere il simbolo # dal relativo blocco indicato nella funzione
#    simula_portafoglio_addetto().

SHOCKS = {
    # ---------------------------------------------------------------
    # DOMANDA E CLIENTI
    # ---------------------------------------------------------------

    # Variazione del numero di clienti attivi durante l'anno.
    "variazione_clienti_pct": {
        "mean": 0.00,
        "sd": 0.035,
        "min": -0.12,
        "max": 0.12,
    },

    # Variazione del valore/volume annuo medio ordinato dal cliente.
    "variazione_valore_cliente_pct": {
        "mean": 0.00,
        "sd": 0.08,
        "min": -0.25,
        "max": 0.30,
    },

    # Quote di ordini ridotti, sospesi o non effettuati.
    "ordini_in_meno_pct": {
        "mean": 0.025,
        "sd": 0.018,
        "min": 0.00,
        "max": 0.12,
    },

    # Sconti, insoluti, note di credito e rettifiche commerciali.
    "rettifiche_ricavi_pct": {
        "mean": 0.015,
        "sd": 0.010,
        "min": 0.00,
        "max": 0.07,
    },

    # ---------------------------------------------------------------
    # VISITE E PERCORSI
    # ---------------------------------------------------------------

    # Visite aggiuntive dovute a ordini incompleti o dimenticanze.
    "visite_extra_ordini_incompleti_pct": {
        "mean": 0.055,
        "sd": 0.030,
        "min": 0.00,
        "max": 0.20,
    },

    # Visite urgenti generate da ordini tardivi o mancata risposta.
    "visite_urgenti_pct": {
        "mean": 0.025,
        "sd": 0.018,
        "min": 0.00,
        "max": 0.12,
    },

    # Aumento dei km del percorso ordinario per traffico, deviazioni,
    # errori di pianificazione, cantieri e clienti aggiunti.
    "sforamento_km_percorso_pct": {
        "mean": 0.045,
        "sd": 0.035,
        "min": -0.04,
        "max": 0.22,
    },

    # Km incrementali medi per ogni visita non programmata.
    "km_extra_per_visita": {
        "mean": 18.0,
        "sd": 6.0,
        "min": 3.0,
        "max": 45.0,
    },

    # Durata operativa sul posto: consegna, controllo, riordino,
    # pulizia minima e interazione con il referente.
    "minuti_servizio_per_visita": {
        "mean": 24.0,
        "sd": 5.0,
        "min": 10.0,
        "max": 50.0,
    },

    # ---------------------------------------------------------------
    # PERSONALE E CAPACITÀ
    # ---------------------------------------------------------------

    # Giorni annui di indisponibilità dell'addetto per malattia,
    # permessi imprevisti o altre assenze non programmate.
    "giorni_assenza_addetto": {
        "mean": 8.0,
        "sd": 4.0,
        "min": 0.0,
        "max": 30.0,
    },

    # Costo giornaliero per sostituzione, straordinari di colleghi
    # o ricorso a risorse temporanee.
    "costo_sostituzione_giorno": {
        "mean": 185.0,
        "sd": 30.0,
        "min": 100.0,
        "max": 300.0,
    },

    # Costo orario dello straordinario o del sovraccarico remunerato.
    "costo_straordinario_ora": {
        "mean": 28.0,
        "sd": 4.0,
        "min": 18.0,
        "max": 45.0,
    },

    # ---------------------------------------------------------------
    # MACCHINE OCS, GUASTI E FERMI
    # ---------------------------------------------------------------

    # Giorni medi annui di fermo per macchina/cliente.
    "giorni_fermo_macchina_per_cliente": {
        "mean": 2.0,
        "sd": 1.2,
        "min": 0.0,
        "max": 8.0,
    },

    # Quota del valore cliente effettivamente persa durante il fermo.
    # Una parte dei consumi può essere recuperata dopo il ripristino.
    "quota_ricavo_persa_durante_fermo": {
        "mean": 0.65,
        "sd": 0.12,
        "min": 0.20,
        "max": 1.00,
    },

    # Numero annuo di interventi tecnici straordinari per 100 macchine.
    "guasti_straordinari_per_100_macchine": {
        "mean": 9.0,
        "sd": 3.0,
        "min": 1.0,
        "max": 25.0,
    },

    # Costo medio di un intervento tecnico straordinario.
    "costo_medio_intervento_guasto": {
        "mean": 145.0,
        "sd": 40.0,
        "min": 50.0,
        "max": 350.0,
    },

    # ---------------------------------------------------------------
    # COSTI DI PRODOTTO E LOGISTICA
    # ---------------------------------------------------------------

    # Quota del fatturato assorbita da caffè, capsule/cialde,
    # zucchero, bicchieri, palette, latte e altri consumabili.
    "costo_prodotti_pct_ricavi": {
        "mean": 0.41,
        "sd": 0.045,
        "min": 0.28,
        "max": 0.58,
    },

    # Inflazione di caffè e materiali rispetto al costo standard.
    "inflazione_prodotti_pct": {
        "mean": 0.035,
        "sd": 0.025,
        "min": -0.03,
        "max": 0.15,
    },

    # Inflazione del costo/km: carburante, energia, manutenzione,
    # pneumatici e altri costi della mobilità.
    "inflazione_costo_km_pct": {
        "mean": 0.035,
        "sd": 0.030,
        "min": -0.06,
        "max": 0.18,
    },

    # Variazione della manutenzione preventiva per macchina.
    "variazione_manutenzione_pct": {
        "mean": 0.00,
        "sd": 0.12,
        "min": -0.20,
        "max": 0.40,
    },

    # ---------------------------------------------------------------
    # VARIABILI OPZIONALI: RIMUOVERE # PER ATTIVARLE
    # ---------------------------------------------------------------

    # Perdita clienti causata da qualità del servizio insufficiente.
    # "churn_servizio_pct": {
    #     "mean": 0.012,
    #     "sd": 0.010,
    #     "min": 0.00,
    #     "max": 0.06,
    # },

    # Aumento del tempo medio per difficoltà di accesso, parcheggio,
    # registrazione all'ingresso o referente non disponibile.
    # "ritardo_accesso_minuti": {
    #     "mean": 4.0,
    #     "sd": 3.0,
    #     "min": 0.0,
    #     "max": 18.0,
    # },

    # Resi, prodotti scaduti o eccedenze non riutilizzabili.
    # "sprechi_prodotti_pct": {
    #     "mean": 0.010,
    #     "sd": 0.008,
    #     "min": 0.00,
    #     "max": 0.05,
    # },

    # Costo annuo aggiuntivo per sanificazione approfondita,
    # filtri acqua e interventi igienico-tecnici.
    # "costo_extra_igiene_per_macchina": {
    #     "mean": 22.0,
    #     "sd": 8.0,
    #     "min": 5.0,
    #     "max": 55.0,
    # },
}


# =====================================================================
# 4. FUNZIONI DI SUPPORTO
# =====================================================================

def gaussian_troncata(
    rng: np.random.Generator,
    config: dict[str, float],
    size: int,
) -> np.ndarray:
    """
    Estrae da una gaussiana e applica i limiti min/max.

    È una gaussiana troncata per clipping. Non sostituisce un modello
    probabilistico calibrato su dati storici, ma consente un controllo
    chiaro e immediato dell'incertezza.
    """
    media = config["mean"]
    deviazione = config["sd"] * UNCERTAINTY_MULTIPLIER
    valori = rng.normal(media, deviazione, size=size)
    return np.clip(valori, config["min"], config["max"])


def estrai_shocks(
    rng: np.random.Generator,
    n: int,
) -> dict[str, np.ndarray]:
    return {
        nome: gaussian_troncata(rng, configurazione, n)
        for nome, configurazione in SHOCKS.items()
    }


def safe_divide(
    numeratore: np.ndarray,
    denominatore: np.ndarray,
) -> np.ndarray:
    return np.divide(
        numeratore,
        denominatore,
        out=np.zeros_like(numeratore, dtype=float),
        where=denominatore != 0,
    )


# =====================================================================
# 5. SIMULAZIONE DEL PORTAFOGLIO DI UN ADDETTO
# =====================================================================

def simula_portafoglio_addetto(
    configurazione: dict[str, Any],
    shocks_comuni_area: dict[str, np.ndarray],
    rng: np.random.Generator,
    n: int,
) -> pd.DataFrame:
    """
    Simula il portafoglio reale assegnato a un singolo addetto.

    Gli shock economici generali sono comuni all'area:
    - inflazione prodotti;
    - inflazione costo/km.

    Gli altri shock sono estratti specificamente per il portafoglio,
    perché assenze, guasti, visite extra e domanda possono differire.
    """
    shocks = estrai_shocks(rng, n)

    # Sostituzione degli shock comuni regionali.
    shocks["inflazione_prodotti_pct"] = (
        shocks_comuni_area["inflazione_prodotti_pct"]
    )
    shocks["inflazione_costo_km_pct"] = (
        shocks_comuni_area["inflazione_costo_km_pct"]
    )

    clienti_base = configurazione["clienti_base"]
    km_annui_base = configurazione["km_annui_base"]
    valore_cliente_base = configurazione["valore_annuo_cliente_base"]
    frequenza_base = configurazione["frequenza_mensile_base"]
    costo_lavoro_base = configurazione["costo_lavoro_annuo_base"]
    costo_km_base = configurazione["costo_km_base"]
    costo_manutenzione_base = (
        configurazione["costo_manutenzione_macchina_base"]
    )
    macchine_per_cliente = configurazione["macchine_per_cliente"]

    # ---------------------------------------------------------------
    # CLIENTI E RICAVI
    # ---------------------------------------------------------------

    clienti_attivi = np.rint(clienti_base * (1 + shocks["variazione_clienti_pct"]))
    clienti_attivi = np.maximum(clienti_attivi, 1)

    # BLOCCO OPZIONALE: CHURN DA SERVIZIO
    # Rimuovere # dalle righe seguenti dopo avere attivato
    # "churn_servizio_pct" nella configurazione SHOCKS.
    #
    # clienti_persi_servizio = np.rint(
    #     clienti_attivi * shocks["churn_servizio_pct"]
    # )
    # clienti_attivi = np.maximum(
    #     clienti_attivi - clienti_persi_servizio,
    #     1,
    # )

    valore_cliente_lordo = valore_cliente_base * (
        1 + shocks["variazione_valore_cliente_pct"]
    )

    ricavi_lordi = clienti_attivi * valore_cliente_lordo

    perdita_ordini = ricavi_lordi * shocks["ordini_in_meno_pct"]

    macchine_attive = clienti_attivi * macchine_per_cliente

    quota_tempo_fermo = (
        shocks["giorni_fermo_macchina_per_cliente"] / 365
    )
    perdita_ricavi_fermo = (
        ricavi_lordi
        * quota_tempo_fermo
        * shocks["quota_ricavo_persa_durante_fermo"]
    )

    rettifiche_ricavi = (
        ricavi_lordi * shocks["rettifiche_ricavi_pct"]
    )

    ricavi_netti = (
        ricavi_lordi
        - perdita_ordini
        - perdita_ricavi_fermo
        - rettifiche_ricavi
    )
    ricavi_netti = np.maximum(ricavi_netti, 0)

    # ---------------------------------------------------------------
    # VISITE E PERCORSI
    # ---------------------------------------------------------------

    visite_programmate_anno = (
        clienti_attivi * frequenza_base * MESI_ANNO
    )

    visite_extra_incomplete = (
        visite_programmate_anno
        * shocks["visite_extra_ordini_incompleti_pct"]
    )

    visite_urgenti = (
        visite_programmate_anno
        * shocks["visite_urgenti_pct"]
    )

    visite_totali_anno = (
        visite_programmate_anno
        + visite_extra_incomplete
        + visite_urgenti
    )

    visite_mese = visite_totali_anno / MESI_ANNO
    visite_giorno = visite_totali_anno / GIORNI_LAVORATIVI_ANNO

    visite_non_programmate = visite_extra_incomplete + visite_urgenti

    km_percorso_ordinario = (
        km_annui_base
        * (clienti_attivi / clienti_base)
        * (1 + shocks["sforamento_km_percorso_pct"])
    )

    km_visite_extra = (
        visite_non_programmate * shocks["km_extra_per_visita"]
    )

    km_totali = np.maximum(
        km_percorso_ordinario + km_visite_extra,
        0,
    )

    km_per_visita = safe_divide(km_totali, visite_totali_anno)

    # ---------------------------------------------------------------
    # TEMPO, ASSENZE E CAPACITÀ
    # ---------------------------------------------------------------

    minuti_servizio = shocks["minuti_servizio_per_visita"]

    # BLOCCO OPZIONALE: RITARDI DI ACCESSO
    # Rimuovere # dalle righe seguenti dopo avere attivato
    # "ritardo_accesso_minuti" nella configurazione SHOCKS.
    #
    # minuti_servizio = (
    #     minuti_servizio
    #     + shocks["ritardo_accesso_minuti"]
    # )

    ore_servizio = (
        visite_totali_anno * minuti_servizio / 60
    )
    ore_guida = km_totali / VELOCITA_MEDIA_KMH
    ore_richieste = ore_servizio + ore_guida

    giorni_disponibili = np.maximum(
        GIORNI_LAVORATIVI_ANNO
        - shocks["giorni_assenza_addetto"],
        1,
    )
    ore_disponibili = giorni_disponibili * ORE_LAVORATIVE_GIORNO

    ore_straordinario = np.maximum(
        ore_richieste - ore_disponibili,
        0,
    )
    saturazione_operativa = safe_divide(
        ore_richieste,
        ore_disponibili,
    )

    costo_sostituzioni = (
        shocks["giorni_assenza_addetto"]
        * shocks["costo_sostituzione_giorno"]
    )
    costo_straordinari = (
        ore_straordinario
        * shocks["costo_straordinario_ora"]
    )

    # ---------------------------------------------------------------
    # COSTI
    # ---------------------------------------------------------------

    costo_km_effettivo = costo_km_base * (
        1 + shocks["inflazione_costo_km_pct"]
    )
    costo_logistica = km_totali * costo_km_effettivo

    costo_prodotti_ratio = (
        shocks["costo_prodotti_pct_ricavi"]
        * (1 + shocks["inflazione_prodotti_pct"])
    )
    costo_prodotti_ratio = np.clip(
        costo_prodotti_ratio,
        0,
        0.90,
    )
    costo_prodotti = ricavi_netti * costo_prodotti_ratio

    costo_manutenzione_preventiva = (
        macchine_attive
        * costo_manutenzione_base
        * (1 + shocks["variazione_manutenzione_pct"])
    )

    numero_guasti_straordinari = (
        macchine_attive
        * shocks["guasti_straordinari_per_100_macchine"]
        / 100
    )
    costo_guasti = (
        numero_guasti_straordinari
        * shocks["costo_medio_intervento_guasto"]
    )

    # BLOCCO OPZIONALE: SPRECHI DI PRODOTTO
    # Rimuovere # dalle righe seguenti dopo avere attivato
    # "sprechi_prodotti_pct" nella configurazione SHOCKS.
    #
    # costo_sprechi = (
    #     costo_prodotti * shocks["sprechi_prodotti_pct"]
    # )
    #
    # Senza il blocco opzionale:
    costo_sprechi = np.zeros(n)

    # BLOCCO OPZIONALE: COSTI EXTRA IGIENE E FILTRI
    # Rimuovere # dalle righe seguenti dopo avere attivato
    # "costo_extra_igiene_per_macchina" nella configurazione SHOCKS.
    #
    # costo_extra_igiene = (
    #     macchine_attive
    #     * shocks["costo_extra_igiene_per_macchina"]
    # )
    #
    # Senza il blocco opzionale:
    costo_extra_igiene = np.zeros(n)

    costo_personale = (
        costo_lavoro_base
        + costo_sostituzioni
        + costo_straordinari
    )

    costi_operativi_totali = (
        costo_prodotti
        + costo_logistica
        + costo_personale
        + costo_manutenzione_preventiva
        + costo_guasti
        + costo_sprechi
        + costo_extra_igiene
    )

    margine_contribuzione = ricavi_netti - costo_prodotti

    margine_netto_operativo = (
        ricavi_netti - costi_operativi_totali
    )

    roi_operativo_pct = (
        safe_divide(
            margine_netto_operativo,
            costi_operativi_totali,
        )
        * 100
    )

    costo_per_cliente = safe_divide(
        costi_operativi_totali,
        clienti_attivi,
    )
    ricavo_per_cliente = safe_divide(
        ricavi_netti,
        clienti_attivi,
    )
    margine_per_cliente = safe_divide(
        margine_netto_operativo,
        clienti_attivi,
    )
    valore_per_addetto = ricavi_netti

    costo_per_visita = safe_divide(
        costi_operativi_totali,
        visite_totali_anno,
    )
    margine_per_visita = safe_divide(
        margine_netto_operativo,
        visite_totali_anno,
    )

    probabilita_operativa_overload = (
        saturazione_operativa > 1
    )

    dataframe = pd.DataFrame(
        {
            "simulazione": np.arange(1, n + 1),
            "area": configurazione["area"],
            "addetto": configurazione["addetto"],

            # Driver estratti
            "shock_variazione_clienti_pct": (
                shocks["variazione_clienti_pct"]
            ),
            "shock_variazione_valore_cliente_pct": (
                shocks["variazione_valore_cliente_pct"]
            ),
            "shock_ordini_in_meno_pct": (
                shocks["ordini_in_meno_pct"]
            ),
            "shock_visite_extra_pct": (
                shocks["visite_extra_ordini_incompleti_pct"]
            ),
            "shock_visite_urgenti_pct": (
                shocks["visite_urgenti_pct"]
            ),
            "shock_sforamento_km_pct": (
                shocks["sforamento_km_percorso_pct"]
            ),
            "shock_giorni_assenza": (
                shocks["giorni_assenza_addetto"]
            ),
            "shock_giorni_fermo_macchina": (
                shocks["giorni_fermo_macchina_per_cliente"]
            ),
            "shock_inflazione_prodotti_pct": (
                shocks["inflazione_prodotti_pct"]
            ),
            "shock_inflazione_costo_km_pct": (
                shocks["inflazione_costo_km_pct"]
            ),
            "shock_costo_prodotti_pct": (
                shocks["costo_prodotti_pct_ricavi"]
            ),

            # Output operativi
            "clienti_attivi": clienti_attivi,
            "macchine_attive": macchine_attive,
            "visite_programmate_anno": visite_programmate_anno,
            "visite_extra_anno": visite_extra_incomplete,
            "visite_urgenti_anno": visite_urgenti,
            "visite_totali_anno": visite_totali_anno,
            "visite_mese": visite_mese,
            "visite_giorno": visite_giorno,
            "km_totali": km_totali,
            "km_per_visita": km_per_visita,
            "ore_richieste": ore_richieste,
            "ore_disponibili": ore_disponibili,
            "ore_straordinario": ore_straordinario,
            "saturazione_operativa": saturazione_operativa,
            "sovraccarico_operativo": probabilita_operativa_overload,

            # Ricavi e perdite
            "ricavi_lordi": ricavi_lordi,
            "perdita_ordini": perdita_ordini,
            "perdita_ricavi_fermo": perdita_ricavi_fermo,
            "rettifiche_ricavi": rettifiche_ricavi,
            "ricavi_netti": ricavi_netti,

            # Costi
            "costo_prodotti": costo_prodotti,
            "costo_logistica": costo_logistica,
            "costo_personale": costo_personale,
            "costo_manutenzione_preventiva": (
                costo_manutenzione_preventiva
            ),
            "costo_guasti": costo_guasti,
            "costo_sprechi": costo_sprechi,
            "costo_extra_igiene": costo_extra_igiene,
            "costi_operativi_totali": costi_operativi_totali,

            # Risultati economici
            "margine_contribuzione": margine_contribuzione,
            "margine_netto_operativo": margine_netto_operativo,
            "roi_operativo_pct": roi_operativo_pct,
            "valore_per_addetto": valore_per_addetto,
            "ricavo_per_cliente": ricavo_per_cliente,
            "costo_per_cliente": costo_per_cliente,
            "margine_per_cliente": margine_per_cliente,
            "costo_per_visita": costo_per_visita,
            "margine_per_visita": margine_per_visita,
        }
    )

    return dataframe


# =====================================================================
# 6. SIMULAZIONE REGIONALE
# =====================================================================

def simula_area_regionale(
    portafogli: list[dict[str, Any]],
    n: int = N_SIMULATIONS,
    seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Simula tutti gli addetti e aggrega i risultati per area e iterazione.
    """
    rng = np.random.default_rng(seed)

    # Shock macro comuni a tutti gli addetti della stessa area.
    shocks_comuni_area = {
        "inflazione_prodotti_pct": gaussian_troncata(
            rng,
            SHOCKS["inflazione_prodotti_pct"],
            n,
        ),
        "inflazione_costo_km_pct": gaussian_troncata(
            rng,
            SHOCKS["inflazione_costo_km_pct"],
            n,
        ),
    }

    risultati_addetti = []

    for portafoglio in portafogli:
        risultato = simula_portafoglio_addetto(
            configurazione=portafoglio,
            shocks_comuni_area=shocks_comuni_area,
            rng=rng,
            n=n,
        )
        risultati_addetti.append(risultato)

    dettaglio_addetti = pd.concat(
        risultati_addetti,
        ignore_index=True,
    )

    colonne_da_sommare = [
        "clienti_attivi",
        "macchine_attive",
        "visite_programmate_anno",
        "visite_extra_anno",
        "visite_urgenti_anno",
        "visite_totali_anno",
        "km_totali",
        "ore_richieste",
        "ore_disponibili",
        "ore_straordinario",
        "ricavi_lordi",
        "perdita_ordini",
        "perdita_ricavi_fermo",
        "rettifiche_ricavi",
        "ricavi_netti",
        "costo_prodotti",
        "costo_logistica",
        "costo_personale",
        "costo_manutenzione_preventiva",
        "costo_guasti",
        "costo_sprechi",
        "costo_extra_igiene",
        "costi_operativi_totali",
        "margine_contribuzione",
        "margine_netto_operativo",
    ]

    area = (
        dettaglio_addetti
        .groupby(["area", "simulazione"], as_index=False)[
            colonne_da_sommare
        ]
        .sum()
    )

    area["numero_addetti"] = len(portafogli)
    area["clienti_per_addetto"] = (
        area["clienti_attivi"] / area["numero_addetti"]
    )
    area["valore_per_addetto"] = (
        area["ricavi_netti"] / area["numero_addetti"]
    )
    area["visite_mese_per_addetto"] = (
        area["visite_totali_anno"]
        / MESI_ANNO
        / area["numero_addetti"]
    )
    area["visite_giorno_per_addetto"] = (
        area["visite_totali_anno"]
        / GIORNI_LAVORATIVI_ANNO
        / area["numero_addetti"]
    )
    area["km_per_addetto"] = (
        area["km_totali"] / area["numero_addetti"]
    )
    area["km_per_cliente"] = safe_divide(
        area["km_totali"].to_numpy(),
        area["clienti_attivi"].to_numpy(),
    )
    area["km_per_visita"] = safe_divide(
        area["km_totali"].to_numpy(),
        area["visite_totali_anno"].to_numpy(),
    )
    area["costo_per_cliente"] = safe_divide(
        area["costi_operativi_totali"].to_numpy(),
        area["clienti_attivi"].to_numpy(),
    )
    area["ricavo_per_cliente"] = safe_divide(
        area["ricavi_netti"].to_numpy(),
        area["clienti_attivi"].to_numpy(),
    )
    area["margine_per_cliente"] = safe_divide(
        area["margine_netto_operativo"].to_numpy(),
        area["clienti_attivi"].to_numpy(),
    )
    area["roi_operativo_pct"] = (
        safe_divide(
            area["margine_netto_operativo"].to_numpy(),
            area["costi_operativi_totali"].to_numpy(),
        )
        * 100
    )
    area["saturazione_operativa"] = safe_divide(
        area["ore_richieste"].to_numpy(),
        area["ore_disponibili"].to_numpy(),
    )
    area["sovraccarico_operativo"] = (
        area["saturazione_operativa"] > 1
    )

    # Driver medi per iterazione, utili per la sensibilità regionale.
    driver_columns = [
        "shock_variazione_clienti_pct",
        "shock_variazione_valore_cliente_pct",
        "shock_ordini_in_meno_pct",
        "shock_visite_urgenti_pct",
        "shock_sforamento_km_pct",
        "shock_giorni_assenza",
        "shock_giorni_fermo_macchina",
        "shock_inflazione_prodotti_pct",
        "shock_inflazione_costo_km_pct",
        "shock_costo_prodotti_pct",
        "shock_visite_extra_pct",
    ]

    driver_area = (
        dettaglio_addetti
        .groupby(["area", "simulazione"], as_index=False)[driver_columns
        ]
        .mean()
    )

    area = area.merge(
        driver_area,
        on=["area", "simulazione"],
        how="left",
    )

    return dettaglio_addetti, area


# =====================================================================
# 7. SINTESI DEL RISCHIO
# =====================================================================

def riepilogo_distribuzione(
    df: pd.DataFrame,
    gruppo: str,
    indicatori: list[str],
) -> pd.DataFrame:
    righe = []

    for nome_gruppo, dati in df.groupby(gruppo):
        for indicatore in indicatori:
            valori = dati[indicatore].dropna()

            righe.append(
                {
                    gruppo: nome_gruppo,
                    "indicatore": indicatore,
                    "P5": valori.quantile(0.05),
                    "media": valori.mean(),
                    "mediana_P50": valori.median(),
                    "P95": valori.quantile(0.95),
                    "deviazione_standard": valori.std(),
                    "probabilita_valore_negativo": (
                        (valori < 0).mean()
                    ),
                }
            )

    return pd.DataFrame(righe)


def sintesi_addetti(
    dettaglio_addetti: pd.DataFrame,
) -> pd.DataFrame:
    righe = []

    for addetto, dati in dettaglio_addetti.groupby("addetto"):
        righe.append(
            {
                "addetto": addetto,
                "clienti_media": dati["clienti_attivi"].mean(),
                "valore_per_addetto_media": (
                    dati["valore_per_addetto"].mean()
                ),
                "margine_medio": (
                    dati["margine_netto_operativo"].mean()
                ),
                "margine_P5": (
                    dati["margine_netto_operativo"].quantile(0.05)
                ),
                "margine_P50": (
                    dati["margine_netto_operativo"].median()
                ),
                "margine_P95": (
                    dati["margine_netto_operativo"].quantile(0.95)
                ),
                "probabilita_perdita": (
                    dati["margine_netto_operativo"] < 0
                ).mean(),
                "roi_medio_pct": dati["roi_operativo_pct"].mean(),
                "km_medi": dati["km_totali"].mean(),
                "visite_giorno_medie": (
                    dati["visite_giorno"].mean()
                ),
                "saturazione_media": (
                    dati["saturazione_operativa"].mean()
                ),
                "probabilita_sovraccarico": (
                    dati["sovraccarico_operativo"].mean()
                ),
                "costo_cliente_medio": (
                    dati["costo_per_cliente"].mean()
                ),
                "margine_cliente_medio": (
                    dati["margine_per_cliente"].mean()
                ),
            }
        )

    return pd.DataFrame(righe)


def analisi_sensibilita(
    area: pd.DataFrame,
    target: str = "margine_netto_operativo",
) -> pd.DataFrame:
    """
    Correlazione di rango di Spearman tra driver e risultato.

    Non dimostra causalità. Serve a ordinare i fattori associati alle
    variazioni del risultato nella simulazione.
    """
    driver_columns = [
        "shock_variazione_clienti_pct",
        "shock_variazione_valore_cliente_pct",
        "shock_ordini_in_meno_pct",
        "shock_visite_urgenti_pct",
        "shock_sforamento_km_pct",
        "shock_giorni_assenza",
        "shock_giorni_fermo_macchina",
        "shock_inflazione_prodotti_pct",
        "shock_inflazione_costo_km_pct",
        "shock_costo_prodotti_pct",
        "shock_visite_extra_pct",
    ]

    correlazioni = (
        area[driver_columns + [target]]
        .corr(method="spearman")[target]
        .drop(target)
        .sort_values(key=np.abs, ascending=False)
    )

    return (
        correlazioni
        .rename("correlazione_spearman")
        .reset_index()
        .rename(columns={"index": "variabile"})
    )


# =====================================================================
# 8. GRAFICI
# =====================================================================

def grafico_distribuzione_margine(
    area: pd.DataFrame,
    output_dir: Path,
) -> plt.Figure:
    valori = area["margine_netto_operativo"]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(valori, bins=35, alpha=0.75, edgecolor="black")
    ax.axvline(
        valori.mean(),
        linestyle="--",
        label=f"Media: €{valori.mean():,.0f}",
    )
    ax.axvline(
        valori.quantile(0.05),
        linestyle=":",
        label=f"P5: €{valori.quantile(0.05):,.0f}",
    )
    ax.set_title("Distribuzione Monte Carlo del Margine Netto OCS")
    ax.set_xlabel("Margine netto operativo annuo (€)")
    ax.set_ylabel("Numero di simulazioni")
    ax.legend()
    fig.tight_layout()
    return fig


def grafico_distribuzione_roi(
    area: pd.DataFrame,
    output_dir: Path,
) -> plt.Figure:
    valori = area["roi_operativo_pct"]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(valori, bins=35, alpha=0.75, edgecolor="black")
    ax.axvline(
        valori.mean(),
        linestyle="--",
        label=f"Media: {valori.mean():.1f}%",
    )
    ax.axvline(
        valori.quantile(0.05),
        linestyle=":",
        label=f"P5: {valori.quantile(0.05):.1f}%",
    )
    ax.set_title("Distribuzione Monte Carlo del ROI Operativo OCS")
    ax.set_xlabel("ROI operativo (%)")
    ax.set_ylabel("Numero di simulazioni")
    ax.legend()
    fig.tight_layout()
    return fig


def grafico_impatto_km_margine(
    area: pd.DataFrame,
    output_dir: Path,
) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(
        area["km_totali"],
        area["margine_netto_operativo"],
        alpha=0.25,
        s=16,
    )
    ax.set_title("Impatto dei Km Totali sul Margine Netto")
    ax.set_xlabel("Km annui della flotta")
    ax.set_ylabel("Margine netto operativo (€)")
    fig.tight_layout()
    return fig


def grafico_sensibilita(
    sensibilita: pd.DataFrame,
    output_dir: Path,
) -> plt.Figure:
    dati = sensibilita.sort_values(
        "correlazione_spearman",
        ascending=True,
    )

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(
        dati["variabile"],
        dati["correlazione_spearman"],
    )
    ax.axvline(0, linewidth=1)
    ax.set_title(
        "Sensibilità del Margine Netto ai Driver Stocastici"
    )
    ax.set_xlabel(
        "Correlazione di Spearman con il margine"
    )
    ax.set_ylabel("Variabile")
    fig.tight_layout()
    return fig


def grafico_confronto_addetti(
    dettaglio_addetti: pd.DataFrame,
    output_dir: Path,
) -> plt.Figure:
    gruppi = [
        gruppo["margine_netto_operativo"].to_numpy()
        for _, gruppo in dettaglio_addetti.groupby("addetto")
    ]
    etichette = [
        nome
        for nome, _ in dettaglio_addetti.groupby("addetto")
    ]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.boxplot(
        gruppi,
        tick_labels=etichette,
        showfliers=False,
    )
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(
        "Confronto della Distribuzione del Margine per Addetto"
    )
    ax.set_xlabel("Portafoglio/addetto")
    ax.set_ylabel("Margine netto operativo (€)")
    fig.tight_layout()
    return fig


# =====================================================================
# 9. ESPORTAZIONE
# =====================================================================

def esporta_excel(
    dettaglio_addetti: pd.DataFrame,
    area: pd.DataFrame,
    sintesi_area: pd.DataFrame,
    sintesi_per_addetto: pd.DataFrame,
    sensibilita: pd.DataFrame,
    output_dir: Path,
) -> Path:
    percorso = output_dir / "risultati_monte_carlo_ocs.xlsx"

    configurazione_portafogli = pd.DataFrame(PORTAFOGLI_ADDETTI)

    configurazione_shocks = pd.DataFrame(
        [
            {
                "variabile": nome,
                "media": config["mean"],
                "deviazione_standard": config["sd"],
                "minimo": config["min"],
                "massimo": config["max"],
                "moltiplicatore_incertezza": (
                    UNCERTAINTY_MULTIPLIER
                ),
            }
            for nome, config in SHOCKS.items()
        ]
    )

    with pd.ExcelWriter(percorso, engine="openpyxl") as writer:
        configurazione_portafogli.to_excel(
            writer,
            sheet_name="Input_addetti",
            index=False,
        )
        configurazione_shocks.to_excel(
            writer,
            sheet_name="Input_incertezza",
            index=False,
        )
        sintesi_area.to_excel(
            writer,
            sheet_name="Sintesi_area",
            index=False,
        )
        sintesi_per_addetto.to_excel(
            writer,
            sheet_name="Sintesi_addetti",
            index=False,
        )
        sensibilita.to_excel(
            writer,
            sheet_name="Sensibilita",
            index=False,
        )
        area.to_excel(
            writer,
            sheet_name="Simulazioni_area",
            index=False,
        )
        dettaglio_addetti.to_excel(
            writer,
            sheet_name="Simulazioni_addetti",
            index=False,
        )

    return percorso


# =====================================================================
# 10. ESECUZIONE
# =====================================================================

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    dettaglio_addetti, area = simula_area_regionale(
        portafogli=PORTAFOGLI_ADDETTI,
        n=N_SIMULATIONS,
        seed=SEED,
    )

    indicatori_area = [
        "clienti_attivi",
        "ricavi_netti",
        "costi_operativi_totali",
        "margine_netto_operativo",
        "roi_operativo_pct",
        "km_totali",
        "visite_giorno_per_addetto",
        "costo_per_cliente",
        "margine_per_cliente",
        "saturazione_operativa",
    ]

    sintesi_area = riepilogo_distribuzione(
        df=area,
        gruppo="area",
        indicatori=indicatori_area,
    )

    sintesi_per_addetto = sintesi_addetti(
        dettaglio_addetti
    )

    sensibilita = analisi_sensibilita(
        area=area,
        target="margine_netto_operativo",
    )

    grafici = [
        grafico_distribuzione_margine(area, OUTPUT_DIR),
        grafico_distribuzione_roi(area, OUTPUT_DIR),
        grafico_impatto_km_margine(area, OUTPUT_DIR),
        grafico_sensibilita(sensibilita, OUTPUT_DIR),
        grafico_confronto_addetti(
            dettaglio_addetti,
            OUTPUT_DIR,
        ),
    ]

    file_excel = esporta_excel(
        dettaglio_addetti=dettaglio_addetti,
        area=area,
        sintesi_area=sintesi_area,
        sintesi_per_addetto=sintesi_per_addetto,
        sensibilita=sensibilita,
        output_dir=OUTPUT_DIR,
    )

    margine = area["margine_netto_operativo"]
    roi = area["roi_operativo_pct"]

    print("=" * 72)
    print("MONTE CARLO OCS — RISULTATI AREA REGIONALE")
    print("=" * 72)
    print(f"Simulazioni: {N_SIMULATIONS:,}")
    print(f"Area: {area['area'].iloc[0]}")
    print(f"Addetti analizzati: {len(PORTAFOGLI_ADDETTI)}")
    print(
        f"Clienti base: "
        f"{sum(p['clienti_base'] for p in PORTAFOGLI_ADDETTI):,.0f}"
    )
    print()

    print("MARGINE NETTO OPERATIVO")
    print(f"Media:  € {margine.mean():,.2f}")
    print(f"P5:     € {margine.quantile(0.05):,.2f}")
    print(f"Mediana € {margine.median():,.2f}")
    print(f"P95:    € {margine.quantile(0.95):,.2f}")
    print(
        f"Probabilità di perdita: "
        f"{(margine < 0).mean():.2%}"
    )
    print()

    print("ROI OPERATIVO")
    print(f"Media:   {roi.mean():.2f}%")
    print(f"P5:      {roi.quantile(0.05):.2f}%")
    print(f"Mediana: {roi.median():.2f}%")
    print(f"P95:     {roi.quantile(0.95):.2f}%")
    print()

    print(
        "Probabilità di sovraccarico operativo regionale: "
        f"{area['sovraccarico_operativo'].mean():.2%}"
    )
    print()

    print("PRINCIPALI DRIVER DEL MARGINE")
    print(sensibilita.head(8).to_string(index=False))
    print()

    print(f"Excel: {file_excel.resolve()}")
    # Display figures in the notebook
    for fig in grafici:
        display(fig)
        plt.close(fig) # Close the figure to free up memory


if __name__ == "__main__":
    main()
'''

path = Path("/mnt/data/monte_carlo_ocs.py")
path.write_text(code, encoding="utf-8")
print(path)
print(f"Righe: {len(code.splitlines())}")

/mnt/data/monte_carlo_ocs.py
Righe: 1415


In [8]:
from pathlib import Path
path = Path("/mnt/data/monte_carlo_ocs.py")
text = path.read_text(encoding="utf-8")
text = text.replace(
    'return s.rename("correlazione_spearman").reset_index(\n        names="variabile"\n    )',
    'return s.rename("correlazione_spearman").reset_index().rename(columns={"index": "variabile"})'
)
path.write_text(text, encoding="utf-8")
print("corretto")


corretto


In [9]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "/mnt/data/monte_carlo_ocs.py"],
    cwd="/mnt/data",
    capture_output=True,
    text=True,
    timeout=180,
)
print(result.stdout)
print(result.stderr)
print("returncode:", result.returncode)


MONTE CARLO OCS — RISULTATI AREA REGIONALE
Simulazioni: 10,000
Area: Marche
Addetti analizzati: 3
Clienti base: 500

MARGINE NETTO OPERATIVO
Media:  € 13,572.38
P5:     € -28,926.80
Mediana € 13,309.61
P95:    € 56,821.43
Probabilità di perdita: 29.98%

ROI OPERATIVO
Media:   2.47%
P5:      -5.01%
Mediana: 2.35%
P95:     10.21%

Probabilità di sovraccarico operativo regionale: 85.29%

PRINCIPALI DRIVER DEL MARGINE
                          variabile  correlazione_spearman
           shock_costo_prodotti_pct              -0.597559
shock_variazione_valore_cliente_pct               0.588083
      shock_inflazione_prodotti_pct              -0.223141
            shock_sforamento_km_pct              -0.158783
      shock_inflazione_costo_km_pct              -0.147996
             shock_visite_extra_pct              -0.138541
           shock_ordini_in_meno_pct              -0.132896
               shock_giorni_assenza              -0.087074

Excel: /mnt/data/output_monte_carlo_ocs/risultati_